In [1]:
import glob
import math
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

In [2]:
# Download list of Olink genes

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/olink/preprocessed/proteomics_genes.txt

olink_genes = pl.read_csv('proteomics_genes.txt', has_header=False).rename({'column_1': 'region'})
olink_genes

[===========================================================>] Completed 46,102 of 46,102 bytes (100%) /home/dnanexus/ukbgym/utils/average_pheno_per_variant/proteomics_genes.txtt


region
str
"""ENSG00000266967"""
"""ENSG00000114779"""
"""ENSG00000097007"""
"""ENSG00000060971"""
"""ENSG00000157766"""
…
"""ENSG00000173465"""
"""ENSG00000105428"""
"""ENSG00000188372"""


In [ ]:
# Download annotation file for variant-gene mapping
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/annotations_fillna_ukbgym_with_mane.parquet -o /home/dnanexus/data_dir/

anno = pl.read_parquet(
    '/home/dnanexus/data_dir/annotations_fillna_ukbgym_with_mane.parquet', 
    columns=['id', 'region']
)

anno

Error: path "/home/dnanexus/data_dir/annotations_fillna_ukbgym.parquet"
already exists but -f/--overwrite was not set


id,region
str,str
"""chr2:26136528:A:G""","""ENSG00000084733"""
"""chr2:233771145:C:T""","""ENSG00000241119"""
"""chr12:101117325:T:G""","""ENSG00000151572"""
"""chr10:25235985:T:G""","""ENSG00000151025"""
"""chr7:80447426:C:CA""","""ENSG00000135218"""
…,…
"""chr7:90733917:A:C""","""ENSG00000058091"""
"""chr18:754318:C:G""","""ENSG00000176105"""
"""chr3:97895184:G:C""","""ENSG00000080200"""


In [4]:
anno['region'].value_counts(sort=True).join(olink_genes, on='region', how='semi')

region,count
str,u64
"""ENSG00000174469""",605106
"""ENSG00000185008""",444475
"""ENSG00000189283""",435466
"""ENSG00000021645""",385076
"""ENSG00000149972""",376497
…,…
"""ENSG00000179889""",1616
"""ENSG00000160221""",669
"""ENSG00000267368""",611


In [15]:
# prot_file = "cauc_cov_regression_90pcs_prs"  # covariates and PRS corrected
# prot_file = "cauc_protrider_lite_prs_rint"   # PROTRIDER corrected (RINT)
prot_file = "cauc_protrider_lite_prs_t_df"     # PROTRIDER corrected (T distribution)

# Download Olink:
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/olink/adjusted/{prot_file}.parquet -o /home/dnanexus/data_dir/

phenos = pl.read_parquet(f'/home/dnanexus/data_dir/{prot_file}.parquet')

olink_genes_w_data = list(set(phenos.columns).intersection(set(olink_genes['region'])).intersection(set(anno['region'])))

phenos = phenos.select(['sample'] + olink_genes_w_data)

long_phenos = (
    phenos
    .unpivot(
        index='sample',
        on=olink_genes_w_data,
        variable_name='phenotype',
        value_name='pheno_value'
    )

    .with_columns(
        region = pl.col('phenotype'),
        phenotype = pl.col('phenotype') + '_olink',
    )
    .drop_nulls()
)

print(long_phenos['phenotype'].value_counts(sort=True))
long_phenos

Error: path "/home/dnanexus/data_dir/cauc_protrider_lite_prs_t_df.parquet"
already exists but -f/--overwrite was not set
shape: (2_661, 2)
┌───────────────────────┬───────┐
│ phenotype             ┆ count │
│ ---                   ┆ ---   │
│ str                   ┆ u64   │
╞═══════════════════════╪═══════╡
│ ENSG00000108821_olink ┆ 39208 │
│ ENSG00000275718_olink ┆ 39208 │
│ ENSG00000078098_olink ┆ 39208 │
│ ENSG00000163359_olink ┆ 39208 │
│ ENSG00000108578_olink ┆ 39208 │
│ …                     ┆ …     │
│ ENSG00000102837_olink ┆ 31597 │
│ ENSG00000131050_olink ┆ 31502 │
│ ENSG00000111405_olink ┆ 30953 │
│ ENSG00000163131_olink ┆ 30432 │
│ ENSG00000170373_olink ┆ 29106 │
└───────────────────────┴───────┘


sample,phenotype,pheno_value,region
str,str,f64,str
"""5645319""","""ENSG00000004897_olink""",0.349881,"""ENSG00000004897"""
"""5959139""","""ENSG00000004897_olink""",-0.044741,"""ENSG00000004897"""
"""5673208""","""ENSG00000004897_olink""",0.713889,"""ENSG00000004897"""
"""5732867""","""ENSG00000004897_olink""",-1.077274,"""ENSG00000004897"""
"""2074480""","""ENSG00000004897_olink""",-0.277849,"""ENSG00000004897"""
…,…,…,…
"""5533889""","""ENSG00000092098_olink""",-0.344315,"""ENSG00000092098"""
"""3191148""","""ENSG00000092098_olink""",-0.009285,"""ENSG00000092098"""
"""4223555""","""ENSG00000092098_olink""",0.314973,"""ENSG00000092098"""


In [6]:
# Download genotype (long gt) file
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/gt_long.parquet -o /home/dnanexus/data_dir/

long_gt = (
    pl.scan_parquet('/home/dnanexus/data_dir/gt_long.parquet')
    .select(['id', 'sample', 'gt'])
    .filter(pl.col('gt')==1)
)
long_gt.head().collect()

[===========================================================>] Completed 27,974,638,284 of 27,974,638,284 bytes (100%) /home/dnanexus/data_dir/gt_long.parquett


id,sample,gt
str,str,i8
"""chr10:20020006:C:T""","""5317620""",1
"""chr10:20020007:TTTTCTTGC:T""","""5546988""",1
"""chr10:20020010:T:C""","""2793793""",1
"""chr10:20020014:G:A""","""1722739""",1
"""chr10:20020014:G:A""","""1139673""",1


In [ ]:
output_dir = '/home/dnanexus/data_dir/appv_phenos/'
!mkdir -p {output_dir}

CHUNK_SIZE = 100
total_genes = len(olink_genes_w_data)
num_chunks = math.ceil(total_genes / CHUNK_SIZE)

print(f"Processing {total_genes} genes in {num_chunks} chunks...")

# Process in Batches
for i in tqdm(range(0, total_genes, CHUNK_SIZE)):
    # 1. Define the current batch of genes
    chunk_genes = olink_genes_w_data[i : i + CHUNK_SIZE]
    chunk_phenos = [f"{g}_olink" for g in chunk_genes]

# for olink_gene in tqdm(olink_genes_w_data):
    print(f"Processing chunk starting at index: {i}")
    
    (
        anno.filter(pl.col('region').is_in(chunk_genes))
        .lazy()
        .join(
            long_gt,
            on='id',
            how='inner'
        )

        .join(
            long_phenos.filter(pl.col('phenotype').is_in(chunk_phenos)).lazy(),
            on=['sample', 'region'],
            how='inner'
        )

        .group_by(['id', 'region', 'phenotype'])
        .agg(
            n_individuals = pl.len().cast(pl.Int32),
            mean_pheno_value = pl.col('pheno_value').mean().cast(pl.Float32),
            std_pheno_value = pl.col('pheno_value').std().cast(pl.Float32),
        )
        .with_columns(
            mean_pheno_value_rank=pl.col('mean_pheno_value')
                .rank(method="max")
                .cast(pl.Float32),
        )
        .with_columns(
            mean_pheno_value_ptile=(
                pl.col('mean_pheno_value_rank') / pl.len()
            ).cast(pl.Float32),
        )

        .sink_parquet(f'{output_dir}/olink_all_genes_EURunrelated_appv_chunk{i}.parquet')
        # .collect(engine='streaming')
    )

Processing 2661 genes in 27 chunks...


  0%|          | 0/27 [00:00<?, ?it/s]

Processing chunk starting at index: 0


  4%|▎         | 1/27 [00:22<09:52, 22.80s/it]

Processing chunk starting at index: 100


...

 93%|█████████▎| 25/27 [10:07<00:48, 24.28s/it]

Processing chunk starting at index: 2500


 96%|█████████▋| 26/27 [10:33<00:24, 24.56s/it]

Processing chunk starting at index: 2600


100%|██████████| 27/27 [10:50<00:00, 24.08s/it]


## Consolidate parquet

In [8]:
pl.read_parquet(f'{output_dir}/olink_all_genes_EURunrelated_appv_chunk0.parquet')

id,region,phenotype,n_individuals,mean_pheno_value,std_pheno_value,mean_pheno_value_rank,mean_pheno_value_ptile
str,str,str,i32,f32,f32,f32,f32
"""chr10:30616661:C:A""","""ENSG00000151033""","""ENSG00000173702_olink""",4,-0.279341,0.78449,83785.0,0.001694
"""chr10:30615614:A:C""","""ENSG00000151033""","""ENSG00000171236_olink""",59,-0.082708,0.985765,111239.0,0.002249
"""chr10:30618231:G:A""","""ENSG00000151033""","""ENSG00000092853_olink""",1,0.003522,null,124896.0,0.002525
"""chr10:30615857:T:C""","""ENSG00000151033""","""ENSG00000176101_olink""",9,-0.08622,0.791607,110689.0,0.002238
"""chr10:30615857:T:C""","""ENSG00000151033""","""ENSG00000091879_olink""",9,-0.006231,1.000759,123343.0,0.002494
…,…,…,…,…,…,…,…
"""chr10:17825629:G:A""","""ENSG00000260314""","""ENSG00000168685_olink""",1,-0.510816,null,220114.0,0.00445
"""chr10:17825629:G:A""","""ENSG00000260314""","""ENSG00000089234_olink""",1,0.362569,null,639699.0,0.012933
"""chr10:17816669:G:GCGAAGGC""","""ENSG00000260314""","""ENSG00000099256_olink""",1,-0.151791,null,373173.0,0.007545


In [9]:
combined_output_file = f"/home/dnanexus/data_dir/{prot_file}_all_genes_EURunrelated_appv_percentiles.parquet"

# 1. Get list of files manually
files = glob.glob(output_dir)
print(f"Found {len(files)} files.")

# 2. Create a list of LazyFrames
lfs = [pl.scan_parquet(f) for f in files]

# 3. Concatenate with relaxation
combined_lazy = pl.concat(lfs, how="vertical_relaxed")

# 4. Stream to disk
print("Streaming to disk...")
combined_lazy.sink_parquet(combined_output_file, engine='streaming')
print("Done.")

Found 1 files.
Streaming to disk...
Done.


In [10]:
!dx upload {combined_output_file} --path project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/avg_pheno_per_var/

[===========================================================>] Uploaded 27,446,415,143 of 27,446,415,143 bytes (100%) /home/dnanexus/data_dir/cauc_protrider_lite_prs_t_df_all_genes_EURunrelated_appv_percentiles.parquet==========================================================> ] Uploaded 26,776,436,736 of 27,446,415,143 bytes (98%) /home/dnanexus/data_dir/cauc_protrider_lite_prs_t_df_all_genes_EURunrelated_appv_percentiles.parquet                                                         ] Uploaded 1,308,622,848 of 27,446,415,143 bytes (5%) /home/dnanexus/data_dir/cauc_protrider_lite_prs_t_df_all_genes_EURunrelated_appv_percentiles.parquet[=======>                                                    ] Uploaded 3,825,205,248 of 27,446,415,143 bytes (14%) /home/dnanexus/data_dir/cauc_protrider_lite_prs_t_df_all_genes_EURunrelated_appv_percentiles.parquet[============================================>               ] Uploaded 20,736,638,976 of 27,446,415,143 bytes (76%) /home/dnanexus/data_di

In [11]:
a = pl.scan_parquet(combined_output_file)
a.head().collect()

id,region,phenotype,n_individuals,mean_pheno_value,std_pheno_value,mean_pheno_value_rank,mean_pheno_value_ptile
str,str,str,i32,f32,f32,f32,f32
"""chr10:30616661:C:A""","""ENSG00000151033""","""ENSG00000173702_olink""",4,-0.279341,0.78449,83785.0,0.001694
"""chr10:30615614:A:C""","""ENSG00000151033""","""ENSG00000171236_olink""",59,-0.082708,0.985765,111239.0,0.002249
"""chr10:30618231:G:A""","""ENSG00000151033""","""ENSG00000092853_olink""",1,0.003522,null,124896.0,0.002525
"""chr10:30615857:T:C""","""ENSG00000151033""","""ENSG00000176101_olink""",9,-0.08622,0.791607,110689.0,0.002238
"""chr10:30615857:T:C""","""ENSG00000151033""","""ENSG00000091879_olink""",9,-0.006231,1.000759,123343.0,0.002494


In [12]:
a.select(['region']).collect()['region'].unique()

region
str
"""ENSG00000107223"""
"""ENSG00000167114"""
"""ENSG00000112116"""
"""ENSG00000041982"""
"""ENSG00000145920"""
…
"""ENSG00000140319"""
"""ENSG00000151651"""
"""ENSG00000104728"""
